In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage
from scipy import ndimage

Parameters

In [ ]:
filename = 'Leopard'
raw_dimension = 1000

In [ ]:
pores_img = np.fromfile(f"../raw_data/{filename}_pores.raw", dtype=np.int32, sep="")
pores_img = pores_img.reshape([raw_dimension, raw_dimension, raw_dimension])

In [ ]:
pores_img_boundaries = skimage.segmentation.find_boundaries(pores_img, connectivity=1, mode='inner', background=-1)

In [ ]:
plt.imshow(pores_img_boundaries[0])

In [ ]:
%%time
pores_img_boundaries = np.zeros(pores_img.shape)
cnt = 1
pores_boundaries_numbers = {}
more_than_one_neighbour = {}

for i, j, k in np.transpose(np.where(skimage.segmentation.find_boundaries(pores_img, connectivity=1, mode='inner', background=-1))):
    neighbours = []
    directions = [
        (1, 0, 0), (-1, 0, 0),
        (0, 1, 0), (0, -1, 0),
        (0, 0, 1), (0, 0, -1)
    ]
    for dx, dy, dz in directions:
        ni, nj, nk = i + dx, j + dy, k + dz
        # Check that the new coordinates are within the array bounds
        if 0 <= ni < raw_dimension and 0 <= nj < raw_dimension and 0 <= nk < raw_dimension:
            if pores_img[i, j, k] != pores_img[ni, nj, nk] and pores_img[ni, nj, nk] != -1 and pores_img[ni, nj, nk] not in neighbours:
                neighbours.append(pores_img[ni, nj, nk])

    if 0 < len(neighbours) < 2:
        pore_numbers_pair = (pores_img[i, j, k], neighbours[0])
        if pore_numbers_pair not in pores_boundaries_numbers.keys(): # if the pair pore_number1 - pore_number2 is absent, add it. For ascending order the boundary number is positive, for the reverse order it is negative
            pore_numbers_pair_sorted = tuple(sorted(list(pore_numbers_pair)))
            pores_boundaries_numbers[pore_numbers_pair_sorted] = cnt
            pores_boundaries_numbers[(pore_numbers_pair_sorted[1], pore_numbers_pair_sorted[0])] = -cnt
            cnt += 1
        pores_img_boundaries[i, j, k] = pores_boundaries_numbers[pore_numbers_pair]
    elif len(neighbours) > 1:
        for neighbour in neighbours:
            pore_numbers_pair = tuple(sorted([pores_img[i, j, k], neighbour]))
            if pore_numbers_pair not in pores_boundaries_numbers.keys(): # if the pair pore_number1 - pore_number2 is absent, add it. For ascending order the boundary number is positive, for the reverse order it is negative
                pores_boundaries_numbers[pore_numbers_pair] = cnt
                pores_boundaries_numbers[(pore_numbers_pair[1], pore_numbers_pair[0])] = -cnt
                cnt += 1
        more_than_one_neighbour[(i, j, k)] = neighbours
        

In [ ]:
pores_img_boundaries = pores_img_boundaries.astype(np.int32)

In [ ]:
existing_border_indices = ndimage.value_indices(pores_img_boundaries, ignore_value=0)

In [ ]:
import copy

In [ ]:
if 'more_than_one_neighbour_new' not in globals() or len([i for i in more_than_one_neighbour_new if len(more_than_one_neighbour_new[i]) > 1]) != 0:
    print('The block below must be repeated several times until this message stops appearing.')

In [ ]:
more_than_one_neighbour_new = copy.deepcopy(more_than_one_neighbour)
existing_pairs = [i for i in pores_boundaries_numbers.keys() if pores_boundaries_numbers[i] in existing_border_indices.keys()]

cnt = 0

for index in more_than_one_neighbour.keys():
    cnt += 1
    if cnt % 100 == 0:
        print(cnt)
    
    for neighbour in more_than_one_neighbour[index]:
        if (pores_img[index], neighbour) in existing_pairs:
            more_than_one_neighbour_new[index].remove(neighbour)

In [ ]:
for index in more_than_one_neighbour.keys():
    try:
        if len(more_than_one_neighbour_new[index]) == 1:
            pores_img_boundaries[index] = pores_boundaries_numbers[(pores_img[index], more_than_one_neighbour_new[index][0])]
            del more_than_one_neighbour_new[index]
        elif len(more_than_one_neighbour_new[index]) == 0:
            pores_img_boundaries[index] = pores_boundaries_numbers[(pores_img[index], more_than_one_neighbour[index][0])]
            del more_than_one_neighbour_new[index]
    except Exception as e:
        print(e, index)

In [ ]:
more_than_one_neighbour = copy.deepcopy(more_than_one_neighbour_new)

In [ ]:
existing_border_indices = ndimage.value_indices(pores_img_boundaries, ignore_value=0)

In [ ]:
if sorted(existing_border_indices.keys()) == sorted(pores_boundaries_numbers.values()):
    print('All boundaries between pores are marked correctly, none are missing')
else:
    all_boundaries = set(pores_boundaries_numbers.values())
    missing_boundaries = all_boundaries.difference(existing_border_indices.keys())
    print(f'{len(missing_boundaries)} of {len(all_boundaries)} boundaries between pores are missing ({len(missing_boundaries) / len(all_boundaries) * 100:.2f}%)')

In [ ]:
pores_img_boundaries.astype(np.int32).tofile(f'../raw_data/{filename}_pore_boundaries.raw')

In [ ]:
np.save(f'../saved_dicts_and_lists/{filename}_pores_boundaries_numbers.npy', pores_boundaries_numbers)